Main Steps:

Load Dataset
Preprocess Text
Create X and y
One-Hot Encode Labels
Split Train/Test Data
Tokenize Text
Pad Sequences
Build LSTM
Train Model
Evaluate Accuracy
Predict New Tweets

In [2]:
#Data handling
import pandas as pd
import numpy as np

#Visualization
import matplotlib.pyplot as plt

#Text cleaning 
    #(Beautiful Soup 4, which is a Python library used to parse and extract data from HTML and XML documents, as some text might have like this- <p>This product is amazing!</p> Instead of - This product is amazing!)
from bs4 import BeautifulSoup
import re
import string

#Sentiment Analysis
    #(library or its own dictionary)
from textblob import TextBlob 

#NLP with NLTK
import nltk
from nltk.corpus import stopwords
nltk.download('punkt')  #sentence and word tokenizer
nltk.download('wordnet')    #lemmatization support
nltk.download('stopwords')  #stopwords

#Emoji processing
%pip install emoji

import emoji

#Preparing text for neural networks
    #LSTM does not understand text, so we need to convert it into numbers. We will use Tokenizer and pad_sequences from Keras for this purpose.
    #Tokenizer converts text into numbers
    #pad_sequences converts numbers into sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

#Deep Learning Model
from tensorflow.keras.models import Sequential

#Neural Network Layers
from tensorflow.keras.layers import *
from keras.layers import LSTM, Dense, SimpleRNN, Embedding, Flatten, Dropout

#Activation Function
from keras.activations import softmax

#Splitting data
from sklearn.model_selection import train_test_split

#Label Encoding
from tensorflow.keras.utils import to_categorical

#Suppressing warnings
import warnings
warnings.filterwarnings('ignore')





[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ROWTECH\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ROWTECH\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ROWTECH\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# open a zip folder and read the csv containing text to make sentiment analysus using lstm

df = pd.read_csv('sentiment_tweets3.csv.zip',names=['Text','Label'],skiprows=1)
df

,Text,Label
106,just had a real good moment. i missssssssss hi...,0
217,is reading manga http://plurk.com/p/mzp1e,0
220,@comeagainjen http://twitpic.com/2y2lx - http:...,0
288,@lapcat Need to send 'em to my accountant tomo...,0
540,ADD ME ON MYSPACE!!! myspace.com/LookThunder,0
...,...,...
802309,No Depression by G Herbo is my mood from now o...,1
802310,What do you do when depression succumbs the br...,1
802311,Ketamine Nasal Spray Shows Promise Against Dep...,1
802312,dont mistake a bad day with depression! everyo...,1


Data consists of 10314 individual tweets and corresponding labels (1=depression tone, 0=no depression). The goal is to develop a model which can detect the tone of the text pieces, either positive or negative. This is a 2-class classification problem which clearly has counterparts in financial domain; detect market mood from pieces of text.
 \
 Since the input text is human written it needs to be cleaned before it can be utilized in our modelling. We define helper functions for various text cleaning tasks. 
 
First, HTML tags and URLs are removed using these helpers.

In [5]:
def remove_html_tags(text):
    return BeautifulSoup(text,'html.parser').get_text()


def remove_urls(text):
    return re.sub(r'http\S+|www\S+','', text)

Punctuation symbols are removed and common abbreviations are converted to proper text. If this were a financial application then one could consider converting stock tickers to company names, e.g. AAPL to Apple and TSLA to Tesla. This would give model concrete meaning to infer from. Also, to distinguish between "Apple" and "apple", the model should be case-sensitive.

In [6]:
def remove_punctuation(text):
    return text.translate(str.maketrans('','', string.punctuation))

chat_words = {
"AFAIK":"As Far As I Know",
"AFK":"Away From Keyboard",
"ASAP":"As Soon As Possible",
"ATK":"At The Keyboard",
"ATM":"At The Moment",
"A3":"Anytime, Anywhere, Anyplace",
"BAK":"Back At Keyboard",
"BBL":"Be Back Later",
"BBS":"Be Back Soon",
"BFN":"Bye For Now",
"B4N":"Bye For Now",
"BRB":"Be Right Back",
"BRT":"Be Right There",
"BTW":"By The Way",
"B4":"Before",
"B4N":"Bye For Now",
"CU":"See You",
"CUL8R":"See You Later",
"CYA":"See You",
"FAQ":"Frequently Asked Questions",
"FC":"Fingers Crossed",
"FWIW":"For What It's Worth",
"FYI":"For Your Information",
"GAL":"Get A Life",
"GG":"Good Game",
"GN":"Good Night",
"GMTA":"Great Minds Think Alike",
"GR8":"Great!",
"G9":"Genius",
"IC":"I See",
"ICQ":"I Seek you (also a chat program)",
"ILU":"ILU: I Love You",
"IMHO":"In My Honest/Humble Opinion",
"IMO":"In My Opinion",
"IOW":"In Other Words",
"IRL":"In Real Life",
"KISS":"Keep It Simple, Stupid",
"LDR":"Long Distance Relationship",
"LMAO":"Laugh My A.. Off",
"LOL":"Laughing Out Loud",
"LTNS":"Long Time No See",
"L8R":"Later",
"MTE":"My Thoughts Exactly",
"M8":"Mate",
"NRN":"No Reply Necessary",
"OIC":"Oh I See",
"PITA":"Pain In The A..",
"PRT":"Party",
"PRW":"Parents Are Watching",
"QPSA?":"Que Pasa?",
"ROFL":"Rolling On The Floor Laughing",
"ROFLOL":"Rolling On The Floor Laughing Out Loud",
"ROTFLMAO":"Rolling On The Floor Laughing My A.. Off",
"SK8":"Skate",
"STATS":"Your sex and age",
"ASL":"Age, Sex, Location",
"THX":"Thank You",
"TTFN":"Ta-Ta For Now!",
"TTYL":"Talk To You Later",
"U":"You",
"U2":"You Too",
"U4E":"Yours For Ever",
"WB":"Welcome Back",
"WTF":"What The F...",
"WTG":"Way To Go!",
"WUF":"Where Are You From?",
"W8":"Wait...",
"7K":"Sick:-D Laugher",
"TFW":"That feeling when",
"MFW":"My face when",
"MRW":"My reaction when",
"IFYP":"I feel your pain",
"TNTL":"Trying not to laugh",
"JK":"Just kidding",
"IDC":"I don't care",
"ILY":"I love you",
"IMU":"I miss you",
"ADIH":"Another day in hell",
"ZZZ":"Sleeping, bored, tired",
"WYWH":"Wish you were here",
"TIME":"Tears in my eyes",
"BAE":"Before anyone else",
"FIMH":"Forever in my heart",
"BSAAW":"Big smile and a wink",
"BWL":"Bursting with laughter",
"BFF":"Best friends forever",
"CSL":"Can't stop laughing"
}

def replace_chat_words(text):
    words = text.split()
    for i, word in enumerate(words):
        if word.lower()in chat_words:
            words[i] = chat_words[word.lower()]
    return' '.join(words)

In [7]:
#removes stop words (Stop words are very common words that appear frequently in English but usually contribute little to understanding the overall meaning or sentiment of a sentence.

#  E.g. "are", "in", "as", "the", etc. can be seen as stop words

stop_words = set(stopwords.words('english'))
def remove_stopwords(text):
    words = text.split()
    filtered_words = [word for word in words if word.lower() not in stop_words]
    return' '.join(filtered_words)

def remove_emojis(text):
    return emoji.demojize(text)

In [8]:
###preprocessors are now combined into a single data cleaner
def preprocess(df):
    df['Text'] = df['Text'].str.lower()
    df['Text'] = df['Text'].apply(remove_html_tags)
    df['Text'] = df['Text'].apply(remove_urls)
    df['Text'] = df['Text'].apply(remove_punctuation)
    df['Text'] = df['Text'].apply(replace_chat_words)
    df['Text'] = df['Text'].apply(remove_stopwords)
    df['Text'] = df['Text'].apply(remove_emojis)
    return df

df = preprocess(df)
df

,Text,Label
106,real good moment missssssssss much,0
217,reading manga,0
220,comeagainjen,0
288,lapcat need send em accountant tomorrow oddly ...,0
540,add myspace myspacecomlookthunder,0
...,...,...
802309,depression g herbo mood im done stressing peop...,1
802310,depression succumbs brain makes feel like youl...,1
802311,ketamine nasal spray shows promise depression ...,1
802312,dont mistake bad day depression everyone em,1
